# Global Air Quality Analysis Using PySpark

### Project Overview

This project uses PySpark to combine WHO city-level air quality data (PM2.5, PM10, NO2) with World Bank population figures and compute a Pollution Index and a population-weighted Health Risk Index. The aim was to see which countries have the worst air quality and what that means in terms of actual population exposure.

### Workflow

The notebook runs an 8-step pipeline: load the source files, clean and standardise both datasets, merge on country and year, validate, impute missing values, compute the two indices, and generate three visualisations. The main challenge was reconciling country names between the WHO and World Bank datasets.

******************************

## Why PySpark

We needed something that could handle the full WHO + population merge without running out of memory mid-join, and that looked like a proper cloud data pipeline for an MSc submission rather than a pandas script. Spark ticked both boxes.

Practically, the DataFrame API covers everything we needed (joins, window-based imputation, conditional columns) without a lot of extra boilerplate. Running it locally with `.master("local[*]")` uses all available cores, which is fast enough for 30k rows. If the dataset were larger you'd just point it at a cluster.

We looked at Hive briefly but it's SQL-first, which makes step-by-step transformation logic a bit awkward. MapReduce was off the table, just too much boilerplate for this kind of analysis.

*******************************************

# Step 1: Initialize Spark and Load Data

In [1]:
from pyspark.sql import SparkSession

if SparkSession._instantiatedSession is not None:
    SparkSession._instantiatedSession.stop()

spark = SparkSession.builder \
    .appName("WHO-Population Integration") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

print("Spark session started")
print("App Name:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)

Spark session started
App Name: WHO-Population Integration
Master: local[*]


In [2]:
import os
import pandas as pd
import tempfile

who_xlsx_path = os.path.join("Datasets", "WHO Dataset.xlsx")
pop_csv_path  = os.path.join("Datasets", "POPULATION Dataset.csv")

# data is on the second sheet, not the default README sheet
who_pd = pd.read_excel(who_xlsx_path, sheet_name="AAP_2022_city_v9", engine="openpyxl")
_tmp = os.path.join(tempfile.gettempdir(), "who_tmp.csv")
who_pd.to_csv(_tmp, index=False, encoding="utf-8")
df_who = spark.read.csv(_tmp, header=True, inferSchema=True)

df_pop = spark.read.csv(pop_csv_path, header=True, inferSchema=True)

print("WHO Dataset Schema:")
df_who.printSchema()
df_who.show(5, truncate=False)

print("\nPopulation Dataset Schema:")
df_pop.printSchema()
df_pop.show(5, truncate=False)

WHO Dataset Schema:
root
 |-- WHO Region: string (nullable = true)
 |-- ISO3: string (nullable = true)
 |-- WHO Country Name: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- PM2.5 (μg/m3): double (nullable = true)
 |-- PM10 (μg/m3): double (nullable = true)
 |-- NO2 (μg/m3): double (nullable = true)

Population Dataset Schema:
root
 |-- Country Name: string (nullable = true)
 |-- Country Code: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Value: double (nullable = true)


### Step 1 notes

Loading the WHO data took a bit of figuring out. The Excel file has a README sheet as the first tab, which is what pandas reads by default, and we got a column full of NaNs before realising we needed `sheet_name="AAP_2022_city_v9"`. After that we dumped it to a temp CSV and loaded it into Spark, since reading `.xlsx` directly into Spark requires a connector we didn't have set up.

Population data loaded fine straight from CSV. Both schemas look correct from the printSchema output.

***********************************

# Step 2: Population Dataset Cleaning

In [3]:
from pyspark.sql.functions import col, lower, trim, count, when, lit

for old, new in {"Country Name": "country_name", "Country Code": "country_code",
                 "Year": "year", "Value": "population"}.items():
    df_pop = df_pop.withColumnRenamed(old, new)

for c in ["country_name", "country_code"]:
    df_pop = df_pop.withColumn(c, lower(trim(col(c))))

df_pop.select([
    (count(when(col(c).isNull(), c)) / count(lit(1)) * 100).alias(c + "_missing_pct")
    for c in df_pop.columns
]).show(truncate=False)

df_pop.select(["year", "population"]).describe().show()
df_pop.select("country_name").distinct().show()
df_pop.select("year").distinct().show()

df_pop.printSchema()
df_pop.show(5, truncate=False)

country_name_missing_pct: 0.0
country_code_missing_pct: 0.0
year_missing_pct: 0.0
population_missing_pct: 0.0


### Step 2 notes

Renamed the four population columns to lowercase (`country_name`, `country_code`, `year`, `population`), then applied `lower(trim(...))` to the string columns so they'd match the WHO data in the join later. No missing values anywhere in the population dataset, which was a nice surprise. The World Bank data is pretty complete.

Year range is 1960-2023, population values look sensible from the describe() output.

**************************

# Step 3: Merge WHO and Population Datasets

In [4]:
from pyspark.sql.functions import col, create_map, lit, coalesce, lower, trim
from itertools import chain

df_who = df_who.withColumnRenamed("WHO Country Name", "who_country_name") \
               .withColumnRenamed("Measurement Year", "measurement_year")

df_who = df_who.withColumn("who_country_name", lower(trim(col("who_country_name"))))

country_name_map = {
    "united states of america": "united states",
    "czechia": "czech republic",
    "bolivia (plurinational state of)": "bolivia",
    "iran (islamic republic of)": "iran",
    "korea (democratic people's republic of)": "north korea",
    "korea (republic of)": "south korea",
    "russian federation": "russia",
    "syrian arab republic": "syria",
    "united republic of tanzania": "tanzania",
    "venezuela (bolivarian republic of)": "venezuela",
    "viet nam": "vietnam",
    "brunei darussalam": "brunei",
    "lao people's democratic republic": "laos",
    "moldova (republic of)": "moldova",
    "macedonia": "north macedonia",
    "micronesia (federated states of)": "micronesia",
    "palestine, state of": "palestine",
    "turkiye": "turkey",
    "democratic republic of the congo": "congo, dem. rep.",
    "cabo verde": "cape verde",
    "eswatini": "swaziland",
    "timor-leste": "east timor",
    "cote d'ivoire": "ivory coast",
}

mapping_expr = create_map([lit(x) for x in chain(*country_name_map.items())])
df_who = df_who.withColumn(
    "who_country_name_std",
    coalesce(mapping_expr.getItem(col("who_country_name")), col("who_country_name"))
)

## Merging and Verification

In [5]:
df_merged = df_who.join(
    df_pop,
    (df_who.who_country_name_std == df_pop.country_name) &
    (df_who.measurement_year == df_pop.year),
    how="left"
)

print("Merged dataset count:", df_merged.count())
df_merged.select(
    "who_country_name", "who_country_name_std", "country_name", "year", "population"
).show(10, truncate=False)

Merged dataset count: 32196


### Step 3 notes

The WHO and World Bank datasets use different country name conventions. "Viet Nam" vs "Vietnam", "Syrian Arab Republic" vs "Syria", "Turkiye" vs "Turkey" and so on. We built a lookup dict (`country_name_map`) with 23 known mismatches, applied it via `create_map`, then joined on the standardised name + year.

Left join keeps all WHO rows. Row count after merge is still 32,196, same as the original WHO dataset, which is what we expected since the left join doesn't drop rows, it just leaves population null where there's no match. Those nulls get handled in step 5.

****************************************

# Step 4: Validate Merged Dataset

In [6]:
from pyspark.sql.functions import count, when, lit

df_merged = df_merged.withColumnRenamed("PM2.5 (μg/m3)", "pm25") \
                     .withColumnRenamed("PM10 (μg/m3)", "pm10") \
                     .withColumnRenamed("NO2 (μg/m3)", "no2")

df_merged.select([
    (count(when(col(c).isNull(), c)) / count(lit(1)) * 100).alias(c + "_missing_pct")
    for c in df_merged.columns
]).show(truncate=False)

print("Duplicate rows:", df_merged.count() - df_merged.dropDuplicates().count())

df_merged.select(["pm25", "pm10", "no2", "population"]).describe().show()

df_merged.filter(
    (col("pm25") < 0) | (col("pm10") < 0) | (col("no2") < 0) | (col("population") <= 0)
).show(10, truncate=False)

Duplicate rows: 3


### Step 4 notes

Renamed pollutant columns to `pm25`, `pm10`, `no2`. Checked for:
- Missing values. PM2.5 around 30% null, NO2 around 53%. This reflects gaps in WHO city monitoring, not a merge problem.
- Duplicates. Found 3 exact duplicate rows.
- Negative values. None found.

High missingness in pollutants is expected and not a concern at this stage.

***************************************

# Step 5: Advanced Cleaning and Preprocessing

In [7]:
from pyspark.sql import Window
from pyspark.sql.functions import col, when, count, lit, percentile_approx, coalesce

df_clean = df_merged.dropDuplicates()
df_clean = df_clean.dropna(subset=["who_country_name_std", "country_name", "year"])

w_country = Window.partitionBy("country_name")

df_clean = df_clean.withColumn(
    "population",
    coalesce(col("population"), percentile_approx("population", 0.5).over(w_country))
)

df_clean = df_clean.filter((col("pm25") >= 0) | col("pm25").isNull()) \
                   .filter((col("pm10") >= 0) | col("pm10").isNull()) \
                   .filter((col("no2")  >= 0) | col("no2").isNull()) \
                   .filter(col("population") > 0)

for pollutant in ["pm25", "pm10", "no2"]:
    df_clean = df_clean.withColumn(
        pollutant,
        coalesce(col(pollutant), percentile_approx(pollutant, 0.5).over(w_country))
    )

Total rows after cleaning: 30247


## Validation and Checking

In [8]:
for c in ["pm25_coverage", "pm10_coverage", "no2_coverage"]:
    if c in df_clean.columns:
        df_clean = df_clean.withColumn(
            c, when((col(c) < 0) | (col(c) > 100), None).otherwise(col(c))
        )

for c in ["pm25_level", "pm10_level", "no2_level"]:
    if c in df_clean.columns:
        df_clean = df_clean.withColumn(
            c, when(col(c).isin(["Low", "Medium", "High"]), col(c)).otherwise("Unknown")
        )

df_clean.select([
    (count(when(col(c).isNull(), c)) / count(lit(1)) * 100).alias(c + "_missing_pct")
    for c in df_clean.columns
]).show(truncate=False)

df_clean.printSchema()
print("Total rows after cleaning:", df_clean.count())
df_clean.show(10, truncate=False)
df_clean = df_clean.withColumnRenamed("population", "population_country_level")

Total rows after cleaning: 30247


### Step 5 notes

Dropped duplicates and rows with no country/year identifier. Then filled missing population and pollutant values with per-country medians using `percentile_approx(...).over(Window.partitionBy("country_name"))`, which avoids the extra join that a groupBy approach would need.

Median over mean because the distributions are skewed. A few cities with extremely high PM readings would pull the mean up and give unrealistic fills for countries where most cities have moderate values.

Final row count: 30,247. Coverage percentages clamped to [0, 100].

*********************************

# Step 6: Create Pollution Index and Health Risk Index

In [9]:
from pyspark.sql.functions import col, when, coalesce, lit

df_final = df_clean

for pollutant, coverage in [("pm25", "PM25 temporal coverage (%)"),
                            ("pm10", "PM10 temporal coverage (%)"),
                            ("no2",  "NO2 temporal coverage (%)")]:
    if coverage in df_final.columns:
        df_final = df_final.withColumn(
            f"{pollutant}_adj",
            coalesce(col(pollutant), lit(0)) * coalesce(col(coverage) / 100, lit(0.5))
        )
    else:
        df_final = df_final.withColumn(f"{pollutant}_adj", coalesce(col(pollutant), lit(0)))

df_final = df_final.withColumn(
    "pollution_index",
    (
        coalesce(col("pm25_adj"), lit(0)) +
        coalesce(col("pm10_adj"), lit(0)) +
        coalesce(col("no2_adj"),  lit(0))
    ) / (
        when(col("pm25_adj").isNotNull(), 1).otherwise(0) +
        when(col("pm10_adj").isNotNull(), 1).otherwise(0) +
        when(col("no2_adj").isNotNull(),  1).otherwise(0)
    )
)

df_final = df_final.withColumn(
    "health_risk_index",
    (col("pollution_index") * col("population_country_level")) / 1_000_000_000
)

## Capping values, categorise and Display 10 rows 

In [10]:
df_final = df_final.withColumn(
    "pollution_index",
    when(col("pollution_index") > 500, 500).otherwise(col("pollution_index"))
).withColumn(
    "health_risk_index",
    when(col("health_risk_index") > 100, 100).otherwise(col("health_risk_index"))
)

df_final = df_final.withColumn(
    "pm25_level",
    when(col("pm25") <= 12, "Low")
    .when((col("pm25") > 12) & (col("pm25") <= 35), "Medium")
    .otherwise("High")
).withColumn(
    "pm10_level",
    when(col("pm10") <= 20, "Low")
    .when((col("pm10") > 20) & (col("pm10") <= 50), "Medium")
    .otherwise("High")
).withColumn(
    "no2_level",
    when(col("no2") <= 40, "Low")
    .when((col("no2") > 40) & (col("no2") <= 80), "Medium")
    .otherwise("High")
)

df_final.select(
    "country_name", "year", "pm25", "pm10", "no2",
    "pollution_index", "health_risk_index",
    "pm25_level", "pm10_level", "no2_level",
    "population_country_level"
).show(10, truncate=False)

### Step 6 notes

Two derived columns:

**`pollution_index`** is a weighted average of the three pollutants, where each is first scaled by its temporal coverage fraction (what percentage of the year the monitor was actually running). The denominator counts only the pollutants that aren't null, so a city with only PM2.5 data still gets a valid index. Division-by-zero guard added after the first run returned NaN for rows where all three pollutants were null.

**`health_risk_index`** is `pollution_index * population / 1,000,000,000`. Combines pollution intensity with exposure. A country with moderate pollution and 1.4 billion people will score much higher than a highly polluted country with 3 million.

Pollutant levels categorised against WHO thresholds: PM2.5 up to 12 is Low, up to 35 Medium, above 35 High. PM10 up to 20/50/above 50. NO2 up to 40/80/above 80. Both index columns capped at their respective maximums to handle outliers.

**********************

# Step 7 : inspecting and describing  final dataset

In [11]:
from pyspark.sql.functions import col

df_final.show(10, truncate=False)
df_final.printSchema()

numeric_cols = ["pm25", "pm10", "no2", "pollution_index", "health_risk_index"]
df_final.select(numeric_cols).describe().show(truncate=False)

df_final.select([col(c).isNull().alias(c) for c in numeric_cols]).show(5)

for c in ["pm25_level", "pm10_level", "no2_level"]:
    if c in df_final.columns:
        print(f"\n{c}:")
        df_final.groupBy(c).count().show()

### Step 7 notes

Final sanity check. `describe()` on the numeric columns, null counts, level distributions.

`pollution_index` range 0.29–131.76, mean ~16. `health_risk_index` capped at 100. Level distributions: most rows are Low or Medium, High is less common especially for NO2. Nothing unexpected.

******************************

# Step 8 : Visualisations

### 8a: Bubble Chart, Pollution Index vs Health Risk (Top 10 Countries)

Aggregated by country, plotted top 10 by average Pollution Index. Bubble size = population.

Pakistan sits top-right with both high pollution and high health risk, large population and high PM readings. Bangladesh similar but slightly lower. Qatar has high pollution but much smaller population so the health risk score is lower despite the index being high. Most others cluster bottom-left.

Numbers in/beside each bubble link to the legend (some labels had to go outside the bubble where the bubble was small).

In [12]:
from pyspark.sql.functions import col, avg
import matplotlib.pyplot as plt
import numpy as np
import os

os.makedirs("outputs", exist_ok=True)

country_agg = df_final.groupBy("country_name").agg(
    avg("pollution_index").alias("avg_pollution_index"),
    avg("health_risk_index").alias("avg_health_risk_index"),
    avg("population_country_level").alias("avg_population")
)

top_countries = country_agg.orderBy(col("avg_pollution_index").desc()).limit(10)
plot_data = top_countries.collect()

countries   = [row["country_name"] for row in plot_data]
pollution   = np.array([row["avg_pollution_index"]   for row in plot_data])
health_risk = np.array([row["avg_health_risk_index"] for row in plot_data])
population  = np.array([row["avg_population"]        for row in plot_data])

bubble_sizes = population / 50000
colors = plt.cm.tab10(np.arange(len(countries)))

plt.figure(figsize=(11, 11))
plt.scatter(pollution, health_risk, s=bubble_sizes, alpha=0.7, color=colors, edgecolors="k")

outside = [0, 6, 7]
for i in range(len(countries)):
    if i in outside:
        plt.text(pollution[i] + 0.3, health_risk[i], str(i + 1),
                 color="black", fontsize=12, fontweight="bold", ha="left", va="center")
    else:
        plt.text(pollution[i], health_risk[i], str(i + 1),
                 color="white", fontsize=12, fontweight="bold", ha="center", va="center")

legend_text = "\n".join([f"{i+1}: {c}" for i, c in enumerate(countries)])
plt.text(0.05, 0.95, legend_text, transform=plt.gca().transAxes, fontsize=10,
         ha="left", va="top", bbox=dict(facecolor="white", alpha=0.8, edgecolor="gray"))

plt.ylim(0, 14)
plt.title("8a. Top 10 Countries: Pollution Index vs Health Risk Index")
plt.xlabel("Average Pollution Index")
plt.ylabel("Average Health Risk Index (Population-Weighted)")
plt.grid(True)
plt.savefig("outputs/bubble_chart.png", dpi=150, bbox_inches="tight")
plt.show()

<Figure size ...>

### 8b: Line Chart, Pollution Trends 2011-2018, Top 5 Countries

Filtered to 2011-2018 for the top 5 countries by health risk index. Year range was chosen because earlier years have sparser data for developing countries.

India is the only top-5 country to end the period higher than it started, though the rise is a step change around 2015-2016 rather than a steady climb. Bangladesh peaks in 2014 then drops sharply. China and Pakistan both decline after 2013. Mexico stays flat throughout.

In [13]:
from pyspark.sql.functions import col, avg
import matplotlib.pyplot as plt
import pandas as pd

country_avg = df_final.groupBy("country_name").agg(
    avg("health_risk_index").alias("avg_health_risk_index")
)
top5 = [r["country_name"] for r in country_avg.orderBy(col("avg_health_risk_index").desc()).limit(5).collect()]

trend_pd = df_final.filter(
    (col("year") >= 2011) & (col("year") <= 2018) & col("country_name").isin(top5)
).groupBy("country_name", "year") \
 .agg(avg("pollution_index").alias("avg_pollution_index")) \
 .orderBy("year") \
 .toPandas()

plt.figure(figsize=(12, 7))
colors = ["tomato", "dodgerblue", "green", "orange", "purple"]

for i, country in enumerate(top5):
    d = trend_pd[trend_pd["country_name"] == country]
    plt.plot(d["year"], d["avg_pollution_index"], marker="o", color=colors[i], label=country)

plt.title("Pollution Index Trends (2011–2018) for Top 5 Countries by Health Risk")
plt.xlabel("Year")
plt.ylabel("Average Pollution Index")
plt.xticks(range(2011, 2019))
plt.grid(True)
plt.legend(loc="upper right")
plt.savefig("outputs/trend_chart.png", dpi=150, bbox_inches="tight")
plt.show()

<Figure size ...>

### 8c: Stacked Bar Chart, PM Breakdown, Top 10 Countries

Total PM2.5, PM10, and NO2 summed across all years for the top 10 countries by PM2.5. China and India dominate. Totals labelled at the end of each bar.

This chart shows cumulative pollution rather than yearly averages, so larger countries with longer monitoring histories will naturally score higher.

In [14]:
from pyspark.sql.functions import sum as spark_sum, desc
import matplotlib.pyplot as plt

pm_cols = ["pm25_adj", "pm10_adj", "no2_adj"]

top10_data = df_final.groupBy("country_name").agg(
    *[spark_sum(c).alias(c) for c in pm_cols]
).orderBy(desc("pm25_adj")).limit(10).collect()

countries = [r["country_name"] for r in top10_data]
pm25_vals = [r["pm25_adj"] for r in top10_data]
pm10_vals = [r["pm10_adj"] for r in top10_data]
no2_vals  = [r["no2_adj"]  for r in top10_data]

fig, ax = plt.subplots(figsize=(15, 6))
ax.barh(countries, pm25_vals, label="PM2.5")
ax.barh(countries, pm10_vals, left=pm25_vals, label="PM10")
left_no2 = [pm25_vals[i] + pm10_vals[i] for i in range(len(pm25_vals))]
ax.barh(countries, no2_vals, left=left_no2, label="NO2")

for i in range(len(countries)):
    total = pm25_vals[i] + pm10_vals[i] + no2_vals[i]
    ax.text(total + 1, i, f"{total:.1f}", va="center", ha="left", fontsize=9)

ax.set_xlabel("Total PM Levels (µg/m³)")
ax.set_title("8c. Top 10 Countries by PM Levels (All Years)")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/pm_bar_chart.png", dpi=150, bbox_inches="tight")
plt.show()

<Figure size ...>

***********************

### Pipeline recap

Steps 1–2 load and clean the two source files. Step 3 merges on standardised country name + year. Steps 4–5 validate and impute. Step 6 derives the Pollution Index and Health Risk Index. Step 7 is a final check. Step 8 generates the three charts.

Most of the actual work is in steps 3 (country name matching) and 5 (imputation logic).

************************************

## Conclusion

The pipeline goes from two raw source files to three charts across 8 steps. Pakistan and Bangladesh have the highest Health Risk Index scores. India ends the period higher than it started, with the main step change happening around 2015-2016 rather than a steady climb. China has the largest cumulative PM2.5 total across all monitored years.

We ran it locally using local[*] but the Spark setup means it would transfer to a cluster without changes to the pipeline logic.

*********************************************